# CLIP4Outfit — Notebook B1: arquitectura (WTI + extractor de estilo + nivel outfit)

Define los tres niveles del paper (Sección 3) como módulos independientes, y los valida con un
**smoke test sintético** (tensores aleatorios, sin datos reales todavía) para comprobar que las
formas y los gradientes funcionan antes de conectar el pipeline de datos real (Notebook B2).

**Decisiones propias no especificadas en el paper** (documentadas aquí y a repetir en la
memoria):
- Asignación de clusters de K-means bajo `no_grad`, centroides finales recalculados con
  gradiente activo (necesario para que el extractor de estilo sea entrenable).
- Manejo de longitudes variables (`No`, `Nt`) mediante *padding* + máscaras en los tres niveles.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

D_MODEL = 512      # dimensión de CLIP ViT-B/32 (imagen y texto), Sección 4.2 del paper
P_WEIGHT = 0.2      # peso outfit-a-texto en la WTI, óptimo según Tabla 3
ALPHA_STYLE = 0.3   # peso de L_style, óptimo según Fig. 4
BETA_OUTFIT = 0.3   # peso de L_outfit, óptimo según Fig. 4
KO_RATIO = 1 / 3    # ratio de clusters de estilo para el outfit (Sección 4.2)
KT_RATIO = 1 / 6    # ratio de clusters de estilo para el texto (Sección 4.2)
TAU_SCALE = 100.0   # escala de temperatura -- fija, NO aprendible (a diferencia del CLIP del Notebook A)

## 1. Utilidades de máscara y *padding*

`pad_and_mask` convierte una lista de tensores de longitud variable `(N_i, D)` en un batch
`(B, N_max, D)` + una máscara booleana `(B, N_max)` -- se reutiliza en los tres niveles.

In [ ]:
def pad_and_mask(tensor_list):
    """tensor_list: lista de B tensores, cada uno (N_i, D) con N_i variable.
    Devuelve: padded (B, N_max, D), mask (B, N_max) bool -- True donde hay dato real."""
    B = len(tensor_list)
    D = tensor_list[0].shape[-1]
    n_max = max(t.shape[0] for t in tensor_list)
    device_ = tensor_list[0].device
    dtype_ = tensor_list[0].dtype

    padded = torch.zeros(B, n_max, D, device=device_, dtype=dtype_)
    mask = torch.zeros(B, n_max, dtype=torch.bool, device=device_)
    for i, t in enumerate(tensor_list):
        n = t.shape[0]
        padded[i, :n] = t
        mask[i, :n] = True
    return padded, mask


def masked_softmax(logits, mask, dim=-1):
    """logits, mask misma forma. Posiciones con mask=False -> probabilidad 0."""
    logits = logits.masked_fill(~mask, float("-inf"))
    probs = F.softmax(logits, dim=dim)
    probs = torch.nan_to_num(probs, nan=0.0)  # por si alguna fila queda toda enmascarada
    return probs

## 2. Módulo WTI (Weighted Token-wise Interaction) -- Ecuación 1 del paper

Calcula la matriz completa `S (B_outfits, B_textos)` de un batch: cada outfit contra cada texto
del batch (no solo la pareja correcta), que es lo que necesita la pérdida InfoNCE de cada nivel.

`AdaptiveWeightNet` implementa `f_o = Softmax(linear_o(e_o))` y `f_t = Softmax(linear_t(e_t))` --
una capa lineal + softmax enmascarado sobre las posiciones reales (no el *padding*).

In [ ]:
class AdaptiveWeightNet(nn.Module):
    """Una capa lineal que produce UN escalar por token/ítem, seguido de softmax enmascarado."""
    def __init__(self, d_model):
        super().__init__()
        self.linear = nn.Linear(d_model, 1)

    def forward(self, x, mask):
        """x: (B, N, D). mask: (B, N) bool. Devuelve pesos (B, N) que suman 1 por fila."""
        logits = self.linear(x).squeeze(-1)  # (B, N)
        return masked_softmax(logits, mask, dim=-1)


def wti_similarity_batch(item_emb, item_mask, text_emb, text_mask, fo_weight, ft_weight, p=P_WEIGHT):
    """
    item_emb: (B, No, D) L2-normalizado -- embeddings de PRENDA (uno por ítem del outfit)
    item_mask: (B, No) bool
    text_emb: (B, Nt, D) L2-normalizado -- embeddings de PALABRA del texto de consulta
    text_mask: (B, Nt) bool
    fo_weight: (B, No) -- salida de AdaptiveWeightNet sobre item_emb
    ft_weight: (B, Nt) -- salida de AdaptiveWeightNet sobre text_emb

    Devuelve: S (B, B) -- S[n, m] = similitud WTI entre el outfit n y el texto m del batch
    (para TODAS las combinaciones, no solo la diagonal -- lo necesita el InfoNCE de la pérdida).
    """
    # similitud cruzada completa: (B_outfits, No, B_textos, Nt)
    sim = torch.einsum("nid,mjd->nmij", item_emb, text_emb)

    # máscara conjunta: (B_o, B_t, No, Nt) -- True solo donde AMBOS lados son reales (no padding)
    joint_mask = item_mask.unsqueeze(1).unsqueeze(-1) & text_mask.unsqueeze(0).unsqueeze(2)
    sim = sim.masked_fill(~joint_mask, float("-inf"))

    # término 1: cada ítem busca su mejor palabra (max sobre j), ponderado por fo_weight
    max_over_tokens = sim.max(dim=3).values                       # (B_o, B_t, No)
    max_over_tokens = torch.nan_to_num(max_over_tokens, neginf=0.0)
    term1 = (fo_weight.unsqueeze(1) * max_over_tokens).sum(dim=2)  # (B_o, B_t)

    # término 2: cada palabra busca su mejor ítem (max sobre i), ponderado por ft_weight
    max_over_items = sim.max(dim=2).values                        # (B_o, B_t, Nt)
    max_over_items = torch.nan_to_num(max_over_items, neginf=0.0)
    term2 = (ft_weight.unsqueeze(0) * max_over_items).sum(dim=2)   # (B_o, B_t)

    S = (term1 + p * term2) / (1 + p)
    return S

## 3. Extractor de estilo -- Transformer (1 capa, sin *positional encoding*) + K-means greedy

`greedy_kmeans_init` implementa el Algoritmo 1 del paper: el primer centroide es el punto más
"periférico" (máxima suma de distancias al resto), los siguientes se eligen maximizando la
distancia mínima a los ya elegidos (*farthest-point sampling*).

`style_extractor` aplica el truco de diferenciabilidad descrito arriba: la asignación de
clusters se decide sin gradiente, los centroides finales se recalculan con gradiente activo.

In [ ]:
class StyleTransformer(nn.Module):
    """Una sola capa de Transformer, SIN positional encoding (el orden no importa aquí,
    Sección 3.2 del paper)."""
    def __init__(self, d_model, n_heads=8, ffn_dim=None, dropout=0.1):
        super().__init__()
        ffn_dim = ffn_dim or d_model * 4
        self.attention = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        """key_padding_mask: (B, N) bool, True donde HAY QUE IGNORAR (convención de PyTorch,
        ojo que es la inversa de nuestras máscaras 'True=real' -- se invierte al llamar)."""
        attn_out, _ = self.attention(x, x, x, key_padding_mask=key_padding_mask, need_weights=False)
        x = self.norm1(x + self.drop1(attn_out))
        x = self.norm2(x + self.drop2(self.ffn(x)))
        return x


@torch.no_grad()
def greedy_kmeans_init(X, k):
    """X: (N, D) SIN gradiente. Algoritmo 1 del paper. Devuelve (k, D) centroides iniciales."""
    N = X.shape[0]
    dists = torch.cdist(X, X)                      # (N, N)
    first_idx = dists.sum(dim=1).argmax().item()
    chosen = [first_idx]

    for _ in range(1, k):
        centroid_stack = X[chosen]                             # (c, D)
        d_to_centroids = torch.cdist(X, centroid_stack)         # (N, c)
        min_d = d_to_centroids.min(dim=1).values                # (N,)
        min_d[chosen] = -float("inf")                           # no reelegir ya elegidos
        next_idx = min_d.argmax().item()
        chosen.append(next_idx)

    return X[chosen].clone()


@torch.no_grad()
def kmeans_assign(X, k, n_iters=5):
    """X: (N, D) SIN gradiente. Devuelve assign (N,) long -- id de cluster en [0, k)."""
    N = X.shape[0]
    centroids = greedy_kmeans_init(X, k)
    assign = torch.zeros(N, dtype=torch.long, device=X.device)
    for _ in range(n_iters):
        d = torch.cdist(X, centroids)          # (N, k)
        assign = d.argmin(dim=1)
        for c in range(k):
            members = X[assign == c]
            if members.shape[0] > 0:
                centroids[c] = members.mean(dim=0)
    return assign


def style_extractor(X, k_ratio):
    """X: (N, D) CON gradiente -- embeddings ya transformados por StyleTransformer, para UNA
    sola muestra (no batcheado, porque k depende de N que varía por muestra).
    Devuelve: centroides (k_eff, D) CON gradiente (k_eff <= k por si N es muy pequeño)."""
    N = X.shape[0]
    if N <= 1:
        return X
    k = max(1, int(k_ratio * N))
    k = min(k, N)

    assign = kmeans_assign(X.detach(), k)  # decisión discreta, sin gradiente

    centroids = []
    for c in range(k):
        members = X[assign == c]           # indexar el tensor CON gradiente por la asignación fija
        if members.shape[0] > 0:
            centroids.append(members.mean(dim=0))
    if not centroids:
        centroids = [X.mean(dim=0)]
    out = torch.stack(centroids)
    return F.normalize(out, p=2, dim=-1)  # renormalizar -- StyleTransformer no preserva norma L2 unitaria

## 4. Nivel outfit -- Transformer (1 capa) + *mean pooling* -- Ecuación 4 del paper

In [ ]:
class OutfitLevelEncoder(nn.Module):
    """Una sola capa de Transformer + mean pooling, para agregar tokens de estilo en UN vector
    global. Se usa tanto para outfits como para textos (dos instancias separadas, como en el
    paper: F^o_trans y F^t_trans)."""
    def __init__(self, d_model, n_heads=8, dropout=0.1):
        super().__init__()
        self.transformer = StyleTransformer(d_model, n_heads=n_heads, dropout=dropout)

    def forward(self, x, key_padding_mask=None):
        """x: (B, k_max, D) -- tokens de estilo ya paddeados. Devuelve (B, D) via mean pooling
        SOLO sobre las posiciones reales (no el padding)."""
        x = self.transformer(x, key_padding_mask=key_padding_mask)
        real_mask = (~key_padding_mask).unsqueeze(-1).float() if key_padding_mask is not None \
            else torch.ones(x.shape[0], x.shape[1], 1, device=x.device)
        pooled = (x * real_mask).sum(dim=1) / real_mask.sum(dim=1).clamp(min=1.0)
        return F.normalize(pooled, p=2, dim=-1)

## 5. Módulo completo: `CLIP4OutfitCore`

Junta los tres niveles. Recibe embeddings de ítem/palabra YA extraídos por CLIP (eso lo hará el
Notebook B2, que sí toca imágenes reales) -- este módulo empieza justo después de esa
extracción, y devuelve las tres matrices de similitud `S_item`, `S_style`, `S_outfit` de un
batch completo.

In [ ]:
class CLIP4OutfitCore(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=8, dropout=0.1,
                 ko_ratio=KO_RATIO, kt_ratio=KT_RATIO, p_weight=P_WEIGHT):
        super().__init__()
        self.ko_ratio = ko_ratio
        self.kt_ratio = kt_ratio
        self.p_weight = p_weight

        # Nivel ítem: pesos adaptativos de la WTI
        self.fo_item = AdaptiveWeightNet(d_model)
        self.ft_item = AdaptiveWeightNet(d_model)

        # Nivel estilo: transformers de extracción (E^o_trans, E^t_trans) + WTI propia
        self.style_transformer_o = StyleTransformer(d_model, n_heads=n_heads, dropout=dropout)
        self.style_transformer_t = StyleTransformer(d_model, n_heads=n_heads, dropout=dropout)
        self.fo_style = AdaptiveWeightNet(d_model)
        self.ft_style = AdaptiveWeightNet(d_model)

        # Nivel outfit: F^o_trans, F^t_trans + mean pooling
        self.outfit_encoder_o = OutfitLevelEncoder(d_model, n_heads=n_heads, dropout=dropout)
        self.outfit_encoder_t = OutfitLevelEncoder(d_model, n_heads=n_heads, dropout=dropout)

    def forward(self, item_emb, item_mask, text_emb, text_mask):
        """
        item_emb: (B, No_max, D) L2-normalizado -- embeddings de prenda (de CLIP, Notebook B2)
        item_mask: (B, No_max) bool
        text_emb: (B, Nt_max, D) L2-normalizado -- embeddings de palabra (de CLIP, Notebook B2)
        text_mask: (B, Nt_max) bool
        """
        B = item_emb.shape[0]

        # ---------- Nivel ítem ----------
        fo_i = self.fo_item(item_emb, item_mask)
        ft_i = self.ft_item(text_emb, text_mask)
        S_item = wti_similarity_batch(item_emb, item_mask, text_emb, text_mask,
                                       fo_i, ft_i, p=self.p_weight)

        # ---------- Nivel estilo ----------
        # Transformamos (batcheado, sí se puede) y luego extraemos estilo (por muestra, no se puede batchear)
        e_prime_o = self.style_transformer_o(item_emb, key_padding_mask=~item_mask)
        e_prime_t = self.style_transformer_t(text_emb, key_padding_mask=~text_mask)

        style_o_list, style_t_list = [], []
        for b in range(B):
            valid_o = e_prime_o[b][item_mask[b]]   # (No_real_b, D)
            valid_t = e_prime_t[b][text_mask[b]]   # (Nt_real_b, D)
            style_o_list.append(style_extractor(valid_o, self.ko_ratio))
            style_t_list.append(style_extractor(valid_t, self.kt_ratio))

        style_o, style_o_mask = pad_and_mask(style_o_list)
        style_t, style_t_mask = pad_and_mask(style_t_list)

        fo_s = self.fo_style(style_o, style_o_mask)
        ft_s = self.ft_style(style_t, style_t_mask)
        S_style = wti_similarity_batch(style_o, style_o_mask, style_t, style_t_mask,
                                        fo_s, ft_s, p=self.p_weight)

        # ---------- Nivel outfit ----------
        o_vec = self.outfit_encoder_o(style_o, key_padding_mask=~style_o_mask)  # (B, D)
        t_vec = self.outfit_encoder_t(style_t, key_padding_mask=~style_t_mask)  # (B, D)
        S_outfit = o_vec @ t_vec.T  # (B, B) -- coseno directo, ya normalizados

        return S_item, S_style, S_outfit

## 6. Pérdida combinada multi-positivo -- Ecuación 5 del paper + nuestro fix

`L = L_item + 0.3·L_style + 0.3·L_outfit`, aplicando en los TRES niveles el mismo mecanismo
multi-positivo que en la Opción 1 (los pares del batch con texto idéntico son positivos
mutuos, no negativos forzados) -- el problema de duplicidad de outfits existe exactamente igual
aquí. La escala `TAU_SCALE=100` multiplica (convención estándar de CLIP), no divide (que era la
lectura literal, y ambigua, de la ecuación 2 del paper -- decisión ya acordada).

In [ ]:
def masked_nce(logits, positive_mask):
    log_prob = F.log_softmax(logits, dim=1)
    pos_count = positive_mask.sum(dim=1).clamp(min=1.0)
    loss_per_row = -(log_prob * positive_mask).sum(dim=1) / pos_count
    return loss_per_row.mean()


def level_loss(S, group_ids, tau_scale=TAU_SCALE):
    """S: (B, B) similitud sin escalar. group_ids: (B,) -- mismo texto = mismo grupo."""
    logits = S * tau_scale
    positive_mask = (group_ids.unsqueeze(1) == group_ids.unsqueeze(0)).float()
    loss_o2t = masked_nce(logits, positive_mask)
    loss_t2o = masked_nce(logits.T, positive_mask.T)
    return (loss_o2t + loss_t2o) / 2


def clip4outfit_loss(S_item, S_style, S_outfit, group_ids, alpha=ALPHA_STYLE, beta=BETA_OUTFIT):
    L_item = level_loss(S_item, group_ids)
    L_style = level_loss(S_style, group_ids)
    L_outfit = level_loss(S_outfit, group_ids)
    L_total = L_item + alpha * L_style + beta * L_outfit
    return L_total, {"L_item": L_item.item(), "L_style": L_style.item(),
                      "L_outfit": L_outfit.item(), "L_total": L_total.item()}

## 7. *Smoke test* sintético -- validar formas y gradientes antes de tocar datos reales

Generamos outfits y textos de longitud variable con tensores aleatorios (no imágenes/texto de
verdad), pasamos todo por `CLIP4OutfitCore`, calculamos la pérdida, y comprobamos que
`loss.backward()` no revienta y que los gradientes llegan a todos los módulos entrenables.

In [ ]:
torch.manual_seed(0)

B = 6  # batch pequeño para el smoke test
No_range = (4, 19)   # rango real de prendas por outfit, según tu memoria
Nt_range = (3, 15)   # rango razonable de palabras por consulta

# Generamos un par de textos duplicados a propósito, para probar el multi-positivo
texts_debug = ["boho", "casual", "boho", "formal office", "street style", "casual"]
text_to_gid = {}
group_ids = []
for t in texts_debug:
    if t not in text_to_gid:
        text_to_gid[t] = len(text_to_gid)
    group_ids.append(text_to_gid[t])
group_ids = torch.tensor(group_ids, device=device)
print("group_ids (para comprobar el multi-positivo):", group_ids.tolist())

item_list, text_list = [], []
for _ in range(B):
    No = torch.randint(*No_range, (1,)).item()
    Nt = torch.randint(*Nt_range, (1,)).item()
    item_list.append(F.normalize(torch.randn(No, D_MODEL), dim=-1))
    text_list.append(F.normalize(torch.randn(Nt, D_MODEL), dim=-1))

item_emb, item_mask = pad_and_mask(item_list)
text_emb, text_mask = pad_and_mask(text_list)
item_emb, item_mask = item_emb.to(device), item_mask.to(device)
text_emb, text_mask = text_emb.to(device), text_mask.to(device)
item_emb.requires_grad_(True)
text_emb.requires_grad_(True)

print(f"item_emb: {item_emb.shape}   text_emb: {text_emb.shape}")

model = CLIP4OutfitCore().to(device)
S_item, S_style, S_outfit = model(item_emb, item_mask, text_emb, text_mask)
print(f"S_item: {S_item.shape}   S_style: {S_style.shape}   S_outfit: {S_outfit.shape}")

loss, loss_parts = clip4outfit_loss(S_item, S_style, S_outfit, group_ids)
print("Componentes de la pérdida:", loss_parts)

loss.backward()

n_params_with_grad = sum(1 for p in model.parameters() if p.grad is not None)
n_params_total = sum(1 for p in model.parameters())
print(f"Parámetros del modelo con gradiente tras backward: {n_params_with_grad}/{n_params_total}")
print(f"¿item_emb recibió gradiente?  {item_emb.grad is not None and item_emb.grad.abs().sum().item() > 0}")
print(f"¿text_emb recibió gradiente?  {text_emb.grad is not None and text_emb.grad.abs().sum().item() > 0}")

## 8. Comprobaciones adicionales (más allá del *smoke test*)

El *smoke test* de la Sección 7 solo confirma que "no sale humo" (no revienta, las formas
casan, el gradiente llega). Estas cuatro pruebas comprueban que la lógica es correcta, no solo
que no falla.

### 8.1 Sobreajustar un batch minúsculo

Si el modelo no puede llevar la pérdida a casi cero sobre 4 ejemplos fijos en pocos pasos, hay
un error de cableado (pérdida mal conectada, gradiente que no fluye donde debería, optimizador
mal configurado).

In [ ]:
torch.manual_seed(1)

B_tiny = 4
tiny_items = [F.normalize(torch.randn(6, D_MODEL), dim=-1) for _ in range(B_tiny)]
tiny_texts = [F.normalize(torch.randn(5, D_MODEL), dim=-1) for _ in range(B_tiny)]
tiny_item_emb, tiny_item_mask = pad_and_mask(tiny_items)
tiny_text_emb, tiny_text_mask = pad_and_mask(tiny_texts)
tiny_group_ids = torch.arange(B_tiny)  # todos distintos -- caso más simple posible

overfit_model = CLIP4OutfitCore().to(device)
optimizer_tiny = torch.optim.Adam(overfit_model.parameters(), lr=1e-2)  # LR alto a propósito, solo para esta prueba

losses = []
for step in range(60):
    S_i, S_s, S_o = overfit_model(tiny_item_emb, tiny_item_mask, tiny_text_emb, tiny_text_mask)
    loss, parts = clip4outfit_loss(S_i, S_s, S_o, tiny_group_ids)
    optimizer_tiny.zero_grad()
    loss.backward()
    optimizer_tiny.step()
    losses.append(loss.item())

print(f"Loss inicial: {losses[0]:.4f}")
print(f"Loss final (60 pasos):  {losses[-1]:.4f}")
print(f"Reducción: {100 * (1 - losses[-1] / losses[0]):.1f}%")

assert losses[-1] < losses[0] * 0.2, "La pérdida no baja lo suficiente -- revisar el cableado del entrenamiento"
print("\nOK -- el modelo puede sobreajustar un batch trivial, el cableado de la pérdida funciona.")

### 8.2 Invarianza a permutación

Sin *positional encoding*, barajar el orden de las prendas de un outfit (o de las palabras de
un texto) no debería cambiar `S_item`, `S_style` ni `S_outfit`. Usamos el modelo ya entrenado
del *smoke test* (pesos fijos, sin más pasos de gradiente) para que la comparación sea limpia.

In [ ]:
torch.manual_seed(2)
test_model = CLIP4OutfitCore().to(device)
test_model.eval()

No, Nt = 10, 7
item_sample = F.normalize(torch.randn(No, D_MODEL), dim=-1)
text_sample = F.normalize(torch.randn(Nt, D_MODEL), dim=-1)

item_emb_1, item_mask_1 = pad_and_mask([item_sample])
text_emb_1, text_mask_1 = pad_and_mask([text_sample])

perm_items = torch.randperm(No)
perm_texts = torch.randperm(Nt)
item_emb_2, item_mask_2 = pad_and_mask([item_sample[perm_items]])
text_emb_2, text_mask_2 = pad_and_mask([text_sample[perm_texts]])

with torch.no_grad():
    S_i1, S_s1, S_o1 = test_model(item_emb_1, item_mask_1, text_emb_1, text_mask_1)
    S_i2, S_s2, S_o2 = test_model(item_emb_2, item_mask_2, text_emb_2, text_mask_2)

diff_item = (S_i1 - S_i2).abs().max().item()
diff_style = (S_s1 - S_s2).abs().max().item()
diff_outfit = (S_o1 - S_o2).abs().max().item()
print(f"Diferencia máxima tras barajar -- item: {diff_item:.2e}  style: {diff_style:.2e}  outfit: {diff_outfit:.2e}")

TOL = 1e-4
assert diff_item < TOL, "S_item cambia al barajar -- hay una fuga de información de orden"
assert diff_style < TOL, "S_style cambia al barajar -- probablemente en el K-means o el StyleTransformer"
assert diff_outfit < TOL, "S_outfit cambia al barajar"
print("\nOK -- el modelo es invariante al orden, como exige el diseño (sin positional encoding).")

### 8.3 Invarianza a *padding*

Añadir relleno artificial (correctamente enmascarado) a una muestra real no debería cambiar el
resultado frente a la misma muestra sin relleno. Si cambia, hay una máscara que no se está
aplicando en algún punto del *pipeline*.

In [ ]:
torch.manual_seed(3)
No_real, Nt_real = 8, 6
item_sample = F.normalize(torch.randn(No_real, D_MODEL), dim=-1)
text_sample = F.normalize(torch.randn(Nt_real, D_MODEL), dim=-1)

# Versión SIN padding extra (referencia)
item_emb_ref, item_mask_ref = pad_and_mask([item_sample])
text_emb_ref, text_mask_ref = pad_and_mask([text_sample])

# Versión CON padding artificial de más (simulando que hubiera otro outfit más largo en el batch)
item_sample_padded = torch.cat([item_sample, torch.zeros(9, D_MODEL)], dim=0)   # +9 filas de relleno
item_mask_padded_manual = torch.cat([torch.ones(No_real, dtype=torch.bool), torch.zeros(9, dtype=torch.bool)])
text_sample_padded = torch.cat([text_sample, torch.zeros(11, D_MODEL)], dim=0)  # +11 filas de relleno
text_mask_padded_manual = torch.cat([torch.ones(Nt_real, dtype=torch.bool), torch.zeros(11, dtype=torch.bool)])

item_emb_pad = item_sample_padded.unsqueeze(0)
item_mask_pad = item_mask_padded_manual.unsqueeze(0)
text_emb_pad = text_sample_padded.unsqueeze(0)
text_mask_pad = text_mask_padded_manual.unsqueeze(0)

with torch.no_grad():
    S_i1, S_s1, S_o1 = test_model(item_emb_ref, item_mask_ref, text_emb_ref, text_mask_ref)
    S_i2, S_s2, S_o2 = test_model(item_emb_pad, item_mask_pad, text_emb_pad, text_mask_pad)

diff_item = (S_i1 - S_i2).abs().max().item()
diff_style = (S_s1 - S_s2).abs().max().item()
diff_outfit = (S_o1 - S_o2).abs().max().item()
print(f"Diferencia máxima con padding extra -- item: {diff_item:.2e}  style: {diff_style:.2e}  outfit: {diff_outfit:.2e}")

assert diff_item < TOL, "El padding contamina S_item -- revisar la máscara en wti_similarity_batch"
assert diff_style < TOL, "El padding contamina S_style -- revisar la máscara en el extractor de estilo"
assert diff_outfit < TOL, "El padding contamina S_outfit -- revisar la máscara en OutfitLevelEncoder"
print("\nOK -- el padding extra no afecta al resultado, las máscaras funcionan correctamente en los tres niveles.")

### 8.4 Casos límite (`No`/`Nt` mínimos y máximos reales)

Según tu memoria, los outfits de Polyvore Outfit tienen entre 4 y 19 prendas. Probamos ambos
extremos, más una consulta de una sola palabra, y comprobamos que no aparece ningún `NaN`.

In [ ]:
torch.manual_seed(4)


def check_no_nan(No, Nt, label):
    items = [F.normalize(torch.randn(No, D_MODEL), dim=-1) for _ in range(3)]
    texts = [F.normalize(torch.randn(Nt, D_MODEL), dim=-1) for _ in range(3)]
    item_emb, item_mask = pad_and_mask(items)
    text_emb, text_mask = pad_and_mask(texts)
    group_ids = torch.tensor([0, 1, 0])

    with torch.no_grad():
        S_i, S_s, S_o = test_model(item_emb, item_mask, text_emb, text_mask)
    loss, _ = clip4outfit_loss(S_i, S_s, S_o, group_ids)

    has_nan = any(torch.isnan(x).any().item() for x in [S_i, S_s, S_o, loss])
    status = "FALLO (NaN detectado)" if has_nan else "OK"
    print(f"{label:35s} No={No:2d} Nt={Nt:2d} -> {status}")
    assert not has_nan, f"NaN detectado en el caso: {label}"


check_no_nan(No=4,  Nt=6,  label="Outfit mínimo (4 prendas)")
check_no_nan(No=19, Nt=6,  label="Outfit máximo (19 prendas)")
check_no_nan(No=8,  Nt=1,  label="Consulta de una sola palabra")
check_no_nan(No=4,  Nt=1,  label="Ambos extremos a la vez")

print("\nOK -- ningún caso límite genera NaN.")